In [11]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(1, r"..\config")

In [12]:
from paths import DISC_STANDARDS_RAW

raw = pd.read_excel(DISC_STANDARDS_RAW / "EFRAG IG 3 List of ESRS Data Points.xlsx", skiprows=1, sheet_name='ESRS E1')

raw.head()

,ID,ESRS,DR,Paragraph,Related AR,Name,Data Type,Conditional or alternative DP,May \n[V],Appendix B - ESRS 2 \n(SFDR + PILLAR 3 + Benchmark + CL),Appendix C - ESRS 1\nDPs subject to phasing-in provisions applicable to undertaking with less than 750 employees,Appendix C - ESRS 1\nDPs subject to phasing-in provisions applicable to all undertakings
0,E1.GOV-3_01,E1,E1.GOV-3,13,NaN,Disclosure of whether and how climate-related ...,narrative,NaN,NaN,NaN,NaN,NaN
1,E1.GOV-3_02,E1,E1.GOV-3,13,NaN,Percentage of remuneration recognised that is ...,percent,NaN,NaN,NaN,NaN,NaN
2,E1.GOV-3_03,E1,E1.GOV-3,13,NaN,Explanation of climate-related considerations ...,narrative,NaN,NaN,NaN,NaN,NaN
3,E1-1_01,E1,E1-1,14,AR 1,Disclosure of transition plan for climate cha...,narrative,NaN,NaN,CL,NaN,NaN
4,E1-1_02,E1,E1-1,16 a,AR 2,Explanation of how targets are compatible with...,narrative,NaN,NaN,NaN,NaN,NaN


### Converting to JSON

In [13]:
req_keys = [
    'ESRS',
    'DR',
    'Paragraph',
    'Related AR',
    'Name',
    'Data Type',
    'Conditional or alternative DP'
]

filtered_df = raw.loc[:, req_keys]

filfiltered_df =filtered_df.astype(str)

In [14]:
filtered_df =filtered_df.apply(lambda x: x.str.strip())

In [15]:
filtered_df.head()

,ESRS,DR,Paragraph,Related AR,Name,Data Type,Conditional or alternative DP
0,E1,E1.GOV-3,NaN,NaN,Disclosure of whether and how climate-related ...,narrative,NaN
1,E1,E1.GOV-3,NaN,NaN,Percentage of remuneration recognised that is ...,percent,NaN
2,E1,E1.GOV-3,NaN,NaN,Explanation of climate-related considerations ...,narrative,NaN
3,E1,E1-1,NaN,AR 1,Disclosure of transition plan for climate cha...,narrative,NaN
4,E1,E1-1,16 a,AR 2,Explanation of how targets are compatible with...,narrative,NaN


In [16]:
nested = (
    filtered_df.groupby(["ESRS", "DR", "Paragraph"], dropna=False)
      .apply(lambda g: g[["Related AR", "Name", "Data Type", "Conditional or alternative DP"]]
             .rename(columns={
                 "Related AR": "AR",
                 "Data Type": "datatype",
                 "Conditional or alternative DP": "dp_type"
                 })
             .to_dict(orient="records"), include_groups=False)
      .reset_index(name="entries")
)

In [17]:
level1 = (
    nested.groupby("ESRS")
    .apply(lambda g: g.groupby("DR")
           .apply(lambda h: dict(zip(h["Paragraph"], h["entries"])), include_groups=False)
           .to_dict(), include_groups=False)
    .to_dict()
)

- If a property describes the nature, context, or conditions of the connection, it belongs on the relationship.

- If a property describes the inherent identity of the entity, it belongs on the node.

In [8]:
import neo4j

In [9]:
from neo4j import GraphDatabase, ResultSummary

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = "neo4j://localhost"
AUTH = ("neo4j", "neo4j")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [10]:
def run_cypher_query(query: str) -> ResultSummary:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        summary = driver.execute_query(query).summary
    return summary.counters.nodes_created

### Testing Node and Relationship creation

In [5]:

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    summary = driver.execute_query(
        """
        CREATE (Apollo13:Movie {title: 'Apollo 13', tmdbID: 568, released: '1995-06-30', imdbRating: 7.6, genres: ['Drama', 'Adventure', 'IMAX']})
        CREATE (TomH:Person {name: 'Tom Hanks', tmdbID: 31, born: '1956-07-09'})
        CREATE (MegR:Person {name: 'Meg Ryan', tmdbID: 5344, born: '1961-11-19'})
        CREATE (DannyD:Person {name: 'Danny DeVito', tmdbID: 518, born: '1944-11-17'})
        CREATE (JackN:Person {name: 'Jack Nicholson', tmdbID: 514, born: '1937-04-22'})
        CREATE (SleeplessInSeattle:Movie {title: 'Sleepless in Seattle', tmdbID: 858, released: '1993-06-25', imdbRating: 6.8, genres: ['Comedy', 'Drama', 'Romance']})
        CREATE (Hoffa:Movie {title: 'Hoffa', tmdbID: 10410, released: '1992-12-25', imdbRating: 6.6, genres: ['Crime', 'Drama']})
        """
    )

In [16]:
create_relationships = (
    """
    CREATE (Apollo13:Movie {title: 'Apollo 13', tmdbID: 568, released: '1995-06-30', imdbRating: 7.6, genres: ['Drama', 'Adventure', 'IMAX']})
    CREATE (TomH:Person {name: 'Tom Hanks', tmdbID: 31, born: '1956-07-09'})
    CREATE (MegR:Person {name: 'Meg Ryan', tmdbID: 5344, born: '1961-11-19'})
    CREATE (DannyD:Person {name: 'Danny DeVito', tmdbID: 518, born: '1944-11-17'})
    CREATE (JackN:Person {name: 'Jack Nicholson', tmdbID: 514, born: '1937-04-22'})
    CREATE (SleeplessInSeattle:Movie {title: 'Sleepless in Seattle', tmdbID: 858, released: '1993-06-25', imdbRating: 6.8, genres: ['Comedy', 'Drama', 'Romance']})
    CREATE (Hoffa:Movie {title: 'Hoffa', tmdbID: 10410, released: '1992-12-25', imdbRating: 6.6, genres: ['Crime', 'Drama']})
    MERGE (TomH)-[:ACTED_IN]->(Apollo13)
    MERGE (TomH)-[:ACTED_IN]->(SleeplessInSeattle)
    MERGE (MegR)-[:ACTED_IN]->(SleeplessInSeattle)
    MERGE (DannyD)-[:ACTED_IN]->(Hoffa)
    MERGE (DannyD)-[:DIRECTED]->(Hoffa)
    MERGE (JackN)-[:ACTED_IN]->(Hoffa)
    """
)

run_cypher_query(create_relationships)

7

### Converting the ESRS E1 set to Graph

- Make the nodes and relationships in arrows, and generate Cypher query for the same

In [21]:
filtered_df

,ESRS,DR,Paragraph,Related AR,Name,Data Type,Conditional or alternative DP
0,E1,E1.GOV-3,NaN,NaN,Disclosure of whether and how climate-related ...,narrative,NaN
1,E1,E1.GOV-3,NaN,NaN,Percentage of remuneration recognised that is ...,percent,NaN
2,E1,E1.GOV-3,NaN,NaN,Explanation of climate-related considerations ...,narrative,NaN
3,E1,E1-1,NaN,AR 1,Disclosure of transition plan for climate cha...,narrative,NaN
4,E1,E1-1,16 a,AR 2,Explanation of how targets are compatible with...,narrative,NaN
...,...,...,...,...,...,...,...
212,E1,E1-9,68 b,NaN,Disclosure of reconciliations with financial s...,narrative,NaN
213,E1,E1-9,69 a,AR 80,Expected cost savings from climate change miti...,monetary,NaN
214,E1,E1-9,69 a,AR 80,Expected cost savings from climate change adap...,monetary,NaN
215,E1,E1-9,69 b,AR 81,Potential market size of low-carbon products a...,monetary,NaN


In [26]:
from neo4j import GraphDatabase
import pandas as pd

def bulk_import_csv_to_neo4j(driver, batch_size=1000):
    """
    Efficiently import a CSV file into Neo4j using Cypher UNWIND batching.
    
    Expected CSV columns:
      source, target, relationship

    Example:
      CompanyA, CO2 Emissions, REDUCED
      CompanyB, Water Usage, INCREASED
    """

    # Step 1: Read the CSV into memory
    df = filtered_df

    # required_cols = {"source", "target", "relationship"}
    # if not required_cols.issubset(df.columns):
    #     raise ValueError(f"CSV must contain columns: {required_cols}")

    cypher_query = """
      UNWIND $rows AS row
      MERGE (std:Standard {name: row.ESRS})
      MERGE (dr:DisclosureRequirement {name: row.DisclosureRequirement})
      MERGE (ar:ApplicationRequirement {name: row.ApplicationRequirement})
      MERGE (dp:Datapoint {id: row.DatapointID})
      SET dp.name = row.DatapointName,
          dp.type = row.DatapointType,
          dp.unit = row.Unit,
          dp.pillar = row.Pillar
      MERGE (std)-[:HAS_DISCLOSURE]->(dr)
      MERGE (dr)-[:HAS_APPLICATION]->(ar)
      MERGE (ar)-[:HAS_DATAPOINT]->(dp)
      """

    # Step 3: Run in batches
    with driver.session() as session:
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].to_dict("records")
            session.run(cypher_query, {"rows": batch})
            print(f"✅ Inserted rows {i}–{i + len(batch)}")

    driver.close()
    print(f"Imported {len(df)} relationships total into Neo4j.")


In [28]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    bulk_import_csv_to_neo4j(driver)

ClientError: {neo4j_code: Neo.ClientError.Statement.SemanticError} {message: Cannot merge the following node because of null property value for 'name': (:DisclosureRequirement {name: null})} {gql_status: 22G03} {gql_status_description: error: data exception - invalid value type}